In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185, ?

In [ ]:
from sklearn.preprocessing import StandardScaler

import cup_common as cc
import nn_common as nnc
import torch
import torch.nn as nn
import importlib
from functools import partial
import pandas as pd
import copy
from sklearn.model_selection import KFold
import svm_common as sc

importlib.reload(nnc)
importlib.reload(cc)

<h3>Preparation</h3>
<hr/>

<h4>Data preparation</h4>
<p>The train and test dataset are loaded in form of Pandas DataFrame for better interaction with PyTorch.
Eventually there will be these objects:
<ul>
<li>X_tr, y_tr: the train features and train labels (or classes)</li>
<li>X_ts, y_ts: the test features and test labels (or classes)</li>
</ul>
</p>

In [ ]:
importlib.reload(cc)
df_train, df_test = cc.load_set()

# Scaler
scaler = StandardScaler()
# The train set used for model selection
# and final testing
X_tr, y_tr = cc.prepare_dataset(df_train, scaler=scaler, fit_scaler=True)
X_ts, y_ts = cc.prepare_dataset(df_test,  scaler=scaler, fit_scaler=False)

# Important!!!! The following dataset is prepared for investigation purpose and dry run of the NN pipeline,
# it is not absolutely meant to be used for model selection
dry_run_scaler = StandardScaler()
X_tr_dry_run, y_tr_dry_run, X_ts_dry_run, y_ts_dry_run = cc.prepare_dataset_for_dry_run(df_train, ratio=0.2, scaler=dry_run_scaler)

features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

<h4>Network Architecture</h4>
<p>Defining and selecting a hypothesis space with hyper-parameter around a neural network architecture:
<ul>
<li><b>net</b>: represent the internal NN structure with input, hidden and output layer.
One important thing to consider is that the output layer <u>must be kept linear</u> in order to be able to
plug the loss function and the output adapter</li>
<li><b>output_adapter</b>: the function which takes the output from the NN and apply the latest trasformation (classification, regression, etc...).
Must be coherent with the optimizer_template. See function docs for details</li>
<li><b>model_template</b>: the model itself, a skeleton which holds the net structure, and the output adapter function.
The model is always cloned troughout the notebok, just to keep a template reference.</li>
<li><b>optimizer_template</b>: the weight update algorithm</li>
<li><b>scheduler_template</b>: implements adaptive learning rate decay by using ReduceLROnPlateau together with the optimizer template</li>
</ul>
</p>

In [ ]:
# Set to True when model is selcted and assessed
dump_prediction = False

# The seed to be used by data splitter
default_state = 42
default_seed = 1

# Starting learning rate
learning_rate = 1e-3

# The input dimension (i.e. the input layer) is obviously
# the number of feature of the TR set
input_dimension = X_tr.shape[1]


# Defining network architecture
# For simplicity we start with just one hidden layer with small nr of hidden unit
hidden_unit = 64
net = nn.Sequential(
    # 1st Hidden Layer
    nn.Linear(input_dimension, hidden_unit, bias=True), # <- NET
    nn.ReLU(), # <- Activation function
    #nn.Tanh(), # <- Activation function

    # Output layer
    # The output layer is kept linear in order to decouple
    # the network architecture from the choice of the loss function
    # and the prediction phase
    nn.Linear(hidden_unit, 4, bias=True) # <- NET
)

# The output and loss function, must be changed according to the type of task
loss_function = nn.MSELoss()
output_adapter = nnc.regression_mse_adapter()

# Model implementation
model_template = nnc.MLP(net, output_adapter)
# weights initialization (for academic purposes)
model_template.apply(lambda m: nnc.init_weights(m, method="kaiming", nonlinearity="relu"))

# The weight update algorithm
optimizer_template = partial(
    torch.optim.AdamW,
    lr=1e-3,
    weight_decay=1e-4
)

# The factor by learning rate will decrease
learning_rate_decay_factor = 0.5
# The epoch to wait before applying the rate decay factor
learning_rate_decay_patience = 10
# The threshold for measuring the new optimum
learning_rate_decay_threshold = 1e-4

# The learning rate decay function
scheduler_template = partial(
    torch.optim.lr_scheduler.ReduceLROnPlateau,
    mode="min", factor=learning_rate_decay_factor, patience=learning_rate_decay_patience,
    threshold=learning_rate_decay_threshold
)

print(model_template)

<h3>NN Run</h3>
<p>The NN run is separated in two distinct scenarios independent from each other:
<ol>
<li>Hold-out: meant ot be used for inspecting the correctness of the whole pipeline (i.e. no error, effectiveness of learning rate decay, plotting etc...).</li>
<li>KFold: meant to be used for model selection and asessment eventually leading to testing evaluation.</li>
</ol>
</p>
<hr/>

<h4>Hold-out</h4>
<p>Hold-out validation is used for preliminary inspection of learning dynamics, and performed trough a specific dry run set elaborated from the original train set, whereas final model selection based on K-Fold cross-validation.</p>

<h5>Training</h5>
<p>Training the model.</p>

In [ ]:
importlib.reload(nnc)

# custom settings
# Either batch, online or the mini-batch size
ho_batch_size="batch"
ho_epochs=300
# Enabling Early stopping
ho_patience = 50 # patience <0 will disable early stopping
ho_min_delta = 1e-4 # ignored if early stopping disabled

# Clone an untrained model and its optimizer
model_dry_run = copy.deepcopy(model_template)

dry_run_result = nnc.train(
    model_dry_run, (X_tr_dry_run, y_tr_dry_run), optimizer_template, loss_function,
    batch_size=ho_batch_size, epochs=ho_epochs,
    min_delta=ho_min_delta,
    patience=ho_patience,
    seed=default_seed,
    scheduler_template=scheduler_template
)


hist_tr = dry_run_result["hist_tr_loss"]
hist_vl = dry_run_result["hist_vl_loss"]
hist_tr_mee = dry_run_result["hist_tr_mee"]
hist_vl_mee = dry_run_result["hist_vl_mee"]
hist_grad = dry_run_result["hist_grad"]

<h5>Gradient Norm</h5>
<p>The gradient norm should gradually decrease throughout epochs, indicating that the optimization process moves toward a stable region of the loss landscape.</p>

In [ ]:
importlib.reload(nnc)
nnc.plot_gradient_norm_bars(hist_grad, step=10)

<h5>Epoch Loss</h5>

In [ ]:
nnc.plot_epoch_loss(hist_tr, hist_vl)

<h5>MEE</h5>
$$\mathrm{MEE}
=
\frac{1}{N}
\sum_{p=1}^{N}
\left\lVert
\mathbf{o}_p - \mathbf{t}_p
\right\rVert_2
=
\frac{1}{N}
\sum_{p=1}^{N}
\sqrt{
\sum_{k=1}^{4}
(o_{pk} - t_{pk})^2
}$$

In [ ]:
importlib.reload(nnc)
nnc.plot_epoch_mee(hist_tr_mee, hist_vl_mee)

In [ ]:
summary = cc.mee_summary(dry_run_result)
print(summary)

<h5>Dry Run Prediction</h5>
<p>A final hold-out subset was extracted from the training data and used only for didactic prediction analysis, allowing qualitative inspection of the model outputs. <u>This step is not impacting the model selection and did not influence any design or selection decision</u>.</p>

In [ ]:
all_pred_dry_run = nnc.predict(model_dry_run, X_ts_dry_run)

In [ ]:
y_pred_dry_run = pd.DataFrame(all_pred_dry_run[0].detach().cpu().numpy())
sc.plot_true_vs_preds_svr(y_ts_dry_run, y_pred_dry_run)

<hr/>

<h4>KFold</h4>
<p>Model selection is performed using K-Fold cross-validation on the training data, while an initial hold-out split was used only for preliminary analysis and debugging.</p>

<h5>Run</h5>

In [ ]:
importlib.reload(nnc)
# StratifiedKFold when shuffle=True shuffle the samples keeping a balance between them
# Passing a random_state permit reproducible output across multiple function calls
kv_random_state=42
fold_strategy = KFold(n_splits=5, shuffle=True, random_state=kv_random_state)

# Epochs
kv_epochs=500
kv_batch_size=64
# Disable Early stopping
kv_min_delta = 1e-4
kv_patience = -1

inner_train_params = {
    "epochs": kv_epochs,
    "batch_size": kv_batch_size,
    "seed": default_seed,
}

fold_histories = nnc.run_kfold(
    model_template, X_tr, y_tr, optimizer_template, scheduler_template, loss_function, fold_strategy, inner_train_params
)


<h5>Metric</h5>
<p>Main metric used for model assessment.
</p>

In [ ]:
#print(fold_histories[0])
cc.mee_table(fold_histories[0])

In [ ]:
nnc.plot_kfold_bar_vl_mee(fold_histories, use="best")

In [ ]:
importlib.reload(nnc)
nnc.plot_kfold_bar_vl_loss(fold_histories, use="best")

In [ ]:
importlib.reload(nnc)
nnc.plot_kfold_bar_vl_rmse(fold_histories, use="best")

In [ ]:
import numpy as np

best_vals = [h["best_vl_mee"] for h in fold_histories]
mean_m = np.mean(best_vals)
std_m  = np.std(best_vals)
print(mean_m, std_m)

In [ ]:
importlib.reload(nnc)
df_kfold = nnc.kfold_regression_table(
    fold_histories,
    use="best",
    derive_rmse=True
)

<h5>Training</h5>
<p>Model is eventually trained with best parameters verified from KFold, in order to predict on official test set.</p>

In [ ]:
# dopo aver scelto best_params dal KFold
model_kf = copy.deepcopy(model_template)

train_kf_result = nnc.train(
    model_kf, (X_tr, y_tr), optimizer_template, loss_function,
    epochs=inner_train_params['epochs'], batch_size=inner_train_params['batch_size'],scheduler_template=scheduler_template,
    seed=inner_train_params['seed'], patience=inner_train_params.get('patience',0), min_delta=inner_train_params.get('min_delta',None))

hist_kf_tr = train_kf_result["hist_tr_loss"]
hist_kf_vl = train_kf_result["hist_vl_loss"]
hist_kf_grad = train_kf_result["hist_grad"]

In [ ]:
kf_summary = cc.mee_summary(train_kf_result)
print(kf_summary)

<h5>Gradient Norm</h5>
<p>The gradient norm should gradually decrease throughout epochs, indicating that the optimization process moves toward a stable region of the loss landscape.</p>

In [ ]:
nnc.plot_gradient_norm_bars(hist_kf_grad, step=10)

<h5>Epoch Loss</h5>

In [ ]:
nnc.plot_epoch_loss(hist_kf_tr, hist_kf_vl)

<h5>MEE</h5>

In [ ]:
importlib.reload(nnc)
nnc.plot_epoch_mee(hist_tr_mee, hist_vl_mee)

In [ ]:
if dump_prediction:
    all_pred = nnc.predict(model_kf, X_ts)
    print(all_pred)